# Sanskrit to English Neural Machine Translation
**NLU Assignment 2**

This notebook translates Sanskrit sentences to English using **IndicTrans2**
(`ai4bharat/indictrans2-indic-en-dist-200M`), a Transformer **encoder-decoder (seq2seq)**
model that supports Sanskrit (`san_Deva`) to English (`eng_Latn`). We first evaluate it
zero-shot, then fine-tune it on the provided 10,000 training pairs and keep the better model.

**Pre-trained models used (disclosure):**
- IndicTrans2 indic-en distilled 200M (AI4Bharat, MIT license) - the translation model we fine-tune.
- RoBERTa-large - downloaded internally by the `bert-score` library only to compute the BERTScore metric.

All models run locally on the Colab GPU. **No external APIs are used** for translation.

Steps to run: upload `Data-set.zip` in the Files panel, add your HuggingFace token as a
Colab secret named `HF_TOKEN`, select a GPU runtime (T4), then Run all.

## 1. Install dependencies

In [6]:
!pip -q install transformers==4.44.2 IndicTransToolkit==1.1.1 sentencepiece sacremoses nltk==3.9.1 bert-score==0.3.13

## 2. Imports and setup

In [7]:
import os
import re
import time
import random
import zipfile
import unicodedata

import numpy as np
import pandas as pd
import torch

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch:", torch.__version__, "| device:", device)
assert device == "cuda", "Please enable GPU: Runtime -> Change runtime type -> T4 GPU"

torch: 2.11.0+cu128 | device: cuda


## 3. Unzip and load the dataset

In [8]:
DATA_DIR = "/content/Data-set"

if os.path.exists("/content/Data-set.zip"):
    with zipfile.ZipFile("/content/Data-set.zip") as z:
        z.extractall(DATA_DIR)

# For the private test set, just change these two filenames.
FILES = {
    "train_sa": "train_sa_10000.csv", "train_en": "train_en_10000.csv",
    "dev_sa":   "dev_sa_1000.csv",    "dev_en":   "dev_en_1000.csv",
    "test_sa":  "test_sa_1000.csv",   "test_en":  "test_en_1000.csv",
}

def clean_text(t):
    # NFC unicode normalization + collapse extra whitespace
    t = "" if not isinstance(t, str) else t
    t = unicodedata.normalize("NFC", t)
    return re.sub(r"\s+", " ", t).strip()

def load_split(split):
    sa = pd.read_csv(os.path.join(DATA_DIR, FILES[split + "_sa"]), encoding="utf-8-sig")
    en = pd.read_csv(os.path.join(DATA_DIR, FILES[split + "_en"]), encoding="utf-8-sig")
    df = sa.merge(en, on="Source_id").sort_values("Source_id").reset_index(drop=True)
    df["Sentence_sa"] = df["Sentence_sa"].map(clean_text)
    df["Sentence_en"] = df["Sentence_en"].map(clean_text)
    return df

train_df = load_split("train")
dev_df = load_split("dev")
test_df = load_split("test")
print(len(train_df), "train |", len(dev_df), "dev |", len(test_df), "test")

10000 train | 1000 dev | 1000 test


## 4. HuggingFace login
IndicTrans2 is a gated repository, so we log in to download the weights (one-time download,
the model then runs fully locally - this is not a translation API).

In [9]:
from google.colab import userdata
from huggingface_hub import login

login(userdata.get("HF_TOKEN"))

## 5. Load the pre-trained model

In [10]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from IndicTransToolkit.processor import IndicProcessor

MODEL_NAME = "ai4bharat/indictrans2-indic-en-dist-200M"
SRC_LANG, TGT_LANG = "san_Deva", "eng_Latn"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

# fp16 for fast inference; 'eager' attention because the T4 GPU does not support flash-attention 2
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    attn_implementation="eager",
).to(device)
model.eval()

processor = IndicProcessor(inference=True)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

model.SRC:   0%|          | 0.00/3.26M [00:00<?, ?B/s]

model.TGT:   0%|          | 0.00/759k [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

configuration_indictrans.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-indic-en-dist-200M:
- configuration_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_indictrans.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-indic-en-dist-200M:
- modeling_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/913M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

Total parameters: 211,780,608


## 6. Translation and evaluation helpers

In [11]:
BATCH_SIZE = 32
NUM_BEAMS = 5
MAX_LEN = 256

@torch.no_grad()
def translate(sentences):
    translations = []
    for i in range(0, len(sentences), BATCH_SIZE):
        batch = sentences[i : i + BATCH_SIZE]
        batch = processor.preprocess_batch(batch, src_lang=SRC_LANG, tgt_lang=TGT_LANG)
        inputs = tokenizer(batch, truncation=True, padding="longest",
                           max_length=MAX_LEN, return_tensors="pt").to(device)
        output = model.generate(**inputs, use_cache=True, max_length=MAX_LEN, num_beams=NUM_BEAMS)
        decoded = tokenizer.batch_decode(output, skip_special_tokens=True)
        translations.extend(processor.postprocess_batch(decoded, lang=TGT_LANG))
    return translations

In [12]:
from nltk.translate.bleu_score import corpus_bleu, sentence_bleu, SmoothingFunction
from bert_score import score as bert_score

def compute_bleu(hyps, refs):
    # default NLTK BLEU (no custom weights), as required
    hyp_tokens = [h.split() for h in hyps]
    ref_tokens = [[r.split()] for r in refs]
    smooth = SmoothingFunction().method1
    corpus = corpus_bleu(ref_tokens, hyp_tokens)
    sent = np.mean([sentence_bleu(r, h, smoothing_function=smooth)
                    for r, h in zip(ref_tokens, hyp_tokens)])
    return corpus, float(sent)

def compute_bertscore(hyps, refs):
    # F1 BERTScore with rescale_with_baseline=True, as required
    P, R, F1 = bert_score(hyps, refs, lang="en", rescale_with_baseline=True, verbose=False)
    return float(F1.mean())

def evaluate(df, name):
    sources = df["Sentence_sa"].tolist()
    refs = df["Sentence_en"].astype(str).tolist()

    torch.cuda.synchronize()
    start = time.time()
    hyps = translate(sources)
    torch.cuda.synchronize()
    elapsed = time.time() - start

    corpus, sent = compute_bleu(hyps, refs)
    f1 = compute_bertscore(hyps, refs)

    print(f"{name}: corpus BLEU = {corpus:.4f} | avg sentence BLEU = {sent:.4f} | BERTScore F1 = {f1:.4f}")
    print(f"{name}: inference time = {elapsed:.1f}s for {len(sources)} sentences | parameters = {total_params:,}")
    return hyps, corpus, f1, elapsed

## 7. Zero-shot evaluation (before fine-tuning)

In [13]:
_, zs_dev_bleu, _, _ = evaluate(dev_df, "zero-shot dev")
zs_test_hyps, zs_test_bleu, _, _ = evaluate(test_df, "zero-shot test")

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


zero-shot dev: corpus BLEU = 0.1645 | avg sentence BLEU = 0.1523 | BERTScore F1 = 0.5278
zero-shot dev: inference time = 70.5s for 1000 sentences | parameters = 211,780,608


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


zero-shot test: corpus BLEU = 0.1458 | avg sentence BLEU = 0.1379 | BERTScore F1 = 0.5113
zero-shot test: inference time = 62.7s for 1000 sentences | parameters = 211,780,608


## 8. Fine-tune on the provided training data
We fine-tune the whole model for 3 epochs with a small learning rate, using the
10,000 provided sentence pairs only.

In [14]:
from datasets import Dataset
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments

def to_dataset(df):
    src = processor.preprocess_batch(df["Sentence_sa"].tolist(),
                                     src_lang=SRC_LANG, tgt_lang=TGT_LANG)
    return Dataset.from_dict({"src": src, "tgt": df["Sentence_en"].astype(str).tolist()})

def tokenize_fn(examples):
    model_inputs = tokenizer(examples["src"], truncation=True, max_length=MAX_LEN)
    labels = tokenizer(text_target=examples["tgt"], truncation=True, max_length=MAX_LEN)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_ds = to_dataset(train_df).map(tokenize_fn, batched=True, remove_columns=["src", "tgt"])
dev_ds = to_dataset(dev_df).map(tokenize_fn, batched=True, remove_columns=["src", "tgt"])

# training needs fp32 master weights (fp16 weights cannot be unscaled by the AMP optimizer)
model = model.float()
model.config.use_cache = False

args = Seq2SeqTrainingArguments(
    output_dir="/content/finetuned",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    per_device_eval_batch_size=16,
    learning_rate=3e-5,
    num_train_epochs=3,
    warmup_ratio=0.1,
    weight_decay=0.01,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    logging_steps=100,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
    tokenizer=tokenizer,
)
trainer.train()

model.eval()
model.config.use_cache = True
model = model.half()  # back to fp16 for fast inference
print("fine-tuning done")

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss
1,1.788500,1.654629
2,1.510300,1.568331
3,1.398900,1.562511


There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


fine-tuning done


## 9. Evaluation after fine-tuning

In [15]:
_, ft_dev_bleu, _, _ = evaluate(dev_df, "fine-tuned dev")
ft_test_hyps, ft_test_bleu, ft_test_f1, ft_test_time = evaluate(test_df, "fine-tuned test")

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


fine-tuned dev: corpus BLEU = 0.2172 | avg sentence BLEU = 0.2094 | BERTScore F1 = 0.5771
fine-tuned dev: inference time = 103.7s for 1000 sentences | parameters = 211,780,608


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


fine-tuned test: corpus BLEU = 0.2114 | avg sentence BLEU = 0.2076 | BERTScore F1 = 0.5671
fine-tuned test: inference time = 102.6s for 1000 sentences | parameters = 211,780,608


## 10. Write submission.csv
We keep whichever model scored higher on the **dev** set (model selection never looks at test).

In [16]:
if ft_dev_bleu >= zs_dev_bleu:
    final_hyps = ft_test_hyps
    print("using the fine-tuned model")
else:
    final_hyps = zs_test_hyps
    print("using the zero-shot model")

submission = pd.DataFrame({
    "Source_id": test_df["Source_id"],
    "Sentence_en": final_hyps,
})
submission.to_csv("/content/submission.csv", index=False, encoding="utf-8")
print("wrote /content/submission.csv |", len(submission), "rows")
submission.head()

using the fine-tuned model
wrote /content/submission.csv | 1000 rows


,Source_id,Sentence_en
0,1,Eclipse also helps the programmer in error det...
1,2,"""But as it is written in the Scriptures I spea..."
2,3,It will then search for its own driver. I will...
3,4,Iteration for all means that the iterator is s...
4,5,"""And I heard a voice saying The second beast c..."


## 11. Example translations

In [17]:
for i in range(8):
    print("SA :", test_df["Sentence_sa"][i])
    print("REF:", test_df["Sentence_en"][i])
    print("HYP:", final_hyps[i])
    print("-" * 80)

SA : एक्लिप्स् इति प्रोग्रामर् कृते दोषान्वेषणे अपि साहाय्यं करोति।
REF: Eclipse also helps the programmer to find out errors.
HYP: Eclipse also helps the programmer in error detection.
--------------------------------------------------------------------------------
SA : विश्वासकारणादेव समभाषि मया वचः। इति यथा शास्त्रे लिखितं तथैवास्माभिरपि विश्वासजनकम् आत्मानं प्राप्य विश्वासः क्रियते तस्माच्च वचांसि भाष्यन्ते।
REF: "We having the same spirit of faith, according as it is written, I believed, and therefore have I spoken; we also believe, and therefore speak;"
HYP: "But as it is written in the Scriptures I speak these things by faith: and they shall say We have the confidence of the Spirit and shall believe.
--------------------------------------------------------------------------------
SA : तदा, तत्स्वयं ड्रैवर निमित्तम् अन्वेष्यति। अहं 'Cancel' इत्यस्योपरि नुदामि।
REF: Then it will automatically begin searching for drivers. I will click on Cancel.
HYP: It will then search for its own

## 12. Download the submission file

In [18]:
from google.colab import files
files.download("/content/submission.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>